In [2]:
"""ML-First Event Detection Evaluation
Evaluate ML-based event detection against manual ground truth.
No heuristics - pure ML filtering.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys

# Add to path
sys.path.insert(0, '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6')

from beemonitor.processing import EventProcessor
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO

print("="*70)
print("ML-FIRST EVENT DETECTION EVALUATION")
print("="*70)

# Setup
config = Config.default()
#config.models.event_classifier = 'event_classifier_ml_first.pkl'  # Use ML-First model

nest_model = YOLO(config.models.nest_detection)
detector = NestDetector(nest_model, config)

# Paths
input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
#output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/CVPR_Output"
output_folder = "/Users/edwardamoah/Downloads/CVPR_Evaluation_Video_Data_output"
manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'

# Load videos and tracking data
files = [os.path.join(input_data, f) for f in os.listdir(input_data) if 'mp4' in f]
tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.endswith('_tracking_results.csv')]

tracking_data = {}
for file in tracking_files:
    video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
    tracking_data[video_name] = pd.read_csv(file)

print(f"\nDataset:")
print(f"  Videos: {len(files)}")
print(f"  Tracking files: {len(tracking_data)}")

# Load manual ground truth
manual_df = pd.read_csv(manual_csv)
manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()

def parse_manual_time(video, time_str):
    date_part = video.split('_')[1]
    return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")

manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)

print(f"  Manual events: {len(manual_df)}")

def reconstruct_motion_data(tracking_df):
    """Convert tracking CSV to motion_data format."""
    trajectories = []
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        centroids = []
        for _, row in track_data.iterrows():
            centroid_x = (row['x1'] + row['x2']) / 2
            centroid_y = (row['y1'] + row['y2']) / 2
            centroids.append((centroid_x, centroid_y))
        
        bboxes = list(zip(track_data['x1'], track_data['y1'], 
                         track_data['x2'], track_data['y2']))
        frame_numbers = track_data['frame'].tolist()
        
        trajectory = (track_id, centroids, bboxes, frame_numbers)
        trajectories.append(trajectory)
    
    return pd.DataFrame({'tracks': [trajectories]})

# ======================================================================
# PROCESS ALL VIDEOS WITH ML-FIRST
# ======================================================================

print("\n" + "="*70)
print("PROCESSING VIDEOS WITH ML-FIRST DETECTION")
print("="*70)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]
results_by_threshold = {}

for threshold in thresholds:
    print(f"\n{'='*70}")
    print(f"TESTING THRESHOLD = {threshold}")
    print(f"{'='*70}")
    
    processor = EventProcessor(config)
    all_events = []
    per_video_stats = []
    
    for video_name, tracking_df in tracking_data.items():
        # Get video file
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            continue
        video_file = video_file[0]
        
        # Get nests
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  ⚠️  {video_name}: Nest detection failed")
            continue
        
        # Process with ML-First
        motion_data = reconstruct_motion_data(tracking_df)
        
        events = processor.process_tracks(
            motion_data=motion_data,
            nests=nests,
            ml_threshold=threshold
        )
        
        events['video'] = video_name
        all_events.append(events)
        
        # Get manual events for this video
        manual_for_video = len(manual_df[manual_df['video'] == video_name])
        
        per_video_stats.append({
            'video': video_name,
            'detected': len(events),
            'manual': manual_for_video
        })
    
    # Combine all events
    events_df = pd.concat(all_events, ignore_index=True)
    
    print(f"\nTotal events detected: {len(events_df)}")
    print(f"Avg confidence: {events_df['ml_confidence'].mean():.3f}")
    print(f"Confidence range: [{events_df['ml_confidence'].min():.3f}, {events_df['ml_confidence'].max():.3f}]")
    
    # Match to manual ground truth
    def match_events(predicted_events, manual_events, tolerance_sec=3.0):
        """Match predicted events to manual events."""
        matches = []
        used_predicted = set()
        
        for _, man_event in manual_events.iterrows():
            video_name = man_event['video']
            pred_for_video = predicted_events[predicted_events['video'] == video_name]
            
            # Extract video start time
            parts = video_name.split('_')
            date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
            video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            
            for pred_idx, pred_event in pred_for_video.iterrows():
                if pred_idx in used_predicted:
                    continue
                
                # Match action and nest
                pred_nest = int(pred_event['nest'])
                man_nest = int(man_event['nest'])
                
                if (pred_event['action'] == man_event['action'] and pred_nest == man_nest):
                    # Check time difference
                    pred_time = video_start + timedelta(seconds=pred_event['frame_number']/30.0)
                    time_diff = abs(pred_time - man_event['dt'])
                    
                    if time_diff <= timedelta(seconds=tolerance_sec):
                        matches.append({
                            'manual_idx': man_event.name,
                            'predicted_idx': pred_idx,
                            'time_diff': time_diff.total_seconds(),
                            'video': video_name,
                            'action': man_event['action']
                        })
                        used_predicted.add(pred_idx)
                        break
        
        true_positives = len(matches)
        false_positives = len(predicted_events) - true_positives
        false_negatives = len(manual_events) - true_positives
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'tp': true_positives,
            'fp': false_positives,
            'fn': false_negatives,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'matches': matches
        }
    
    metrics = match_events(events_df, manual_df)
    
    # Error analysis by event type
    matched_pred_idx = set([m['predicted_idx'] for m in metrics['matches']])
    matched_manual_idx = set([m['manual_idx'] for m in metrics['matches']])
    
    tp_by_action = {}
    fp_by_action = {}
    fn_by_action = {}
    
    for action in ['Entry', 'Exit']:
        manual_action = manual_df[manual_df['action'] == action]
        tp = sum(1 for m in metrics['matches'] if m['action'] == action)
        fn = len(manual_action) - tp
        
        pred_action = events_df[events_df['action'] == action]
        fp = len(pred_action) - tp
        
        tp_by_action[action] = tp
        fp_by_action[action] = fp
        fn_by_action[action] = fn
    
    # Display results
    print("\n" + "-"*70)
    print("RESULTS")
    print("-"*70)
    
    print(f"\nOverall Performance:")
    print(f"  Predicted events: {len(events_df)}")
    print(f"  Manual events: {len(manual_df)}")
    print(f"  True Positives: {metrics['tp']}")
    print(f"  False Positives: {metrics['fp']}")
    print(f"  False Negatives: {metrics['fn']}")
    
    print(f"\nMetrics:")
    print(f"  Precision: {metrics['precision']:.3f} ({metrics['precision']*100:.1f}%)")
    print(f"  Recall: {metrics['recall']:.3f} ({metrics['recall']*100:.1f}%)")
    print(f"  F1 Score: {metrics['f1']:.3f}")
    
    print(f"\nBy Event Type:")
    for action in ['Entry', 'Exit']:
        manual_count = len(manual_df[manual_df['action'] == action])
        tp = tp_by_action[action]
        fp = fp_by_action[action]
        fn = fn_by_action[action]
        
        recall = tp / manual_count if manual_count > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  {action}:")
        print(f"    Manual: {manual_count}")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")
        print(f"    Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"    Recall: {recall:.3f} ({recall*100:.1f}%)")
    
    # Save results for this threshold
    results_by_threshold[threshold] = {
        'metrics': metrics,
        'events_df': events_df,
        'per_video': pd.DataFrame(per_video_stats)
    }

# ======================================================================
# THRESHOLD COMPARISON
# ======================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON")
print("="*70)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Detected':<12} {'TP':<8} {'FP':<8} {'FN':<8}")
print("-" * 100)

best_f1 = 0
best_threshold = 0.3

for threshold in thresholds:
    m = results_by_threshold[threshold]['metrics']
    detected = len(results_by_threshold[threshold]['events_df'])
    
    print(f"{threshold:<12.1f} {m['precision']:<12.3f} {m['recall']:<12.3f} {m['f1']:<12.3f} {detected:<12} {m['tp']:<8} {m['fp']:<8} {m['fn']:<8}")
    
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_threshold = threshold

print(f"\n✓ Best threshold: {best_threshold} (F1={best_f1:.3f})")

# ======================================================================
# DETAILED RESULTS FOR BEST THRESHOLD
# ======================================================================

print("\n" + "="*70)
print(f"DETAILED RESULTS (Threshold = {best_threshold})")
print("="*70)

best_results = results_by_threshold[best_threshold]
best_metrics = best_results['metrics']
best_events = best_results['events_df']

print(f"\nPer-Video Breakdown:")
print(best_results['per_video'].to_string(index=False))

# Save to CSV
best_results['per_video'].to_csv('ml_first_evaluation_per_video.csv', index=False)
print(f"\n✓ Per-video results saved to ml_first_evaluation_per_video.csv")

# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nML-First Detection:")
print(f"  Window size: 1 frame")
print(f"  Padding: 40 pixels")
print(f"  Minimal trajectory filtering")

print(f"\nBest Configuration:")
print(f"  Threshold: {best_threshold}")
print(f"  Precision: {best_metrics['precision']:.3f} ({best_metrics['precision']*100:.1f}%)")
print(f"  Recall: {best_metrics['recall']:.3f} ({best_metrics['recall']*100:.1f}%)")
print(f"  F1 Score: {best_metrics['f1']:.3f}")

print(f"\nError Analysis:")
print(f"  True Positives: {best_metrics['tp']} (correctly detected)")
print(f"  False Positives: {best_metrics['fp']} (noise detected as events)")
print(f"  False Negatives: {best_metrics['fn']} (missed real events)")

print(f"\nFor CVPR Paper:")
print(f"  \"Our ML-First approach achieves {best_metrics['precision']*100:.1f}% precision")
print(f"   and {best_metrics['recall']*100:.1f}% recall (F1={best_metrics['f1']:.3f})\"")
print(f"  \"No manual parameter tuning - data-driven event detection\"")

print("\n" + "="*70)

ML-FIRST EVENT DETECTION EVALUATION

Dataset:
  Videos: 11
  Tracking files: 11
  Manual events: 300

PROCESSING VIDEOS WITH ML-FIRST DETECTION

TESTING THRESHOLD = 0.3


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 294
Avg confidence: 0.723
Confidence range: [0.302, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 294
  Manual events: 300
  True Positives: 239
  False Positives: 55
  False Negatives: 61

Metrics:
  Precision: 0.813 (81.3%)
  Recall: 0.797 (79.7%)
  F1 Score: 0.805

By Event Type:

  Entry:
    Manual: 149
    TP: 125, FP: 28, FN: 24
    Precision: 0.817 (81.7%)
    Recall: 0.839 (83.9%)

  Exit:
    Manual: 151
    TP: 114, FP: 27, FN: 37
    Precision: 0.809 (80.9%)
    Recall: 0.755 (75.5%)

TESTING THRESHOLD = 0.4


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 250
Avg confidence: 0.790
Confidence range: [0.406, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 250
  Manual events: 300
  True Positives: 212
  False Positives: 38
  False Negatives: 88

Metrics:
  Precision: 0.848 (84.8%)
  Recall: 0.707 (70.7%)
  F1 Score: 0.771

By Event Type:

  Entry:
    Manual: 149
    TP: 113, FP: 19, FN: 36
    Precision: 0.856 (85.6%)
    Recall: 0.758 (75.8%)

  Exit:
    Manual: 151
    TP: 99, FP: 19, FN: 52
    Precision: 0.839 (83.9%)
    Recall: 0.656 (65.6%)

TESTING THRESHOLD = 0.5


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 220
Avg confidence: 0.838
Confidence range: [0.525, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 220
  Manual events: 300
  True Positives: 192
  False Positives: 28
  False Negatives: 108

Metrics:
  Precision: 0.873 (87.3%)
  Recall: 0.640 (64.0%)
  F1 Score: 0.738

By Event Type:

  Entry:
    Manual: 149
    TP: 101, FP: 13, FN: 48
    Precision: 0.886 (88.6%)
    Recall: 0.678 (67.8%)

  Exit:
    Manual: 151
    TP: 91, FP: 15, FN: 60
    Precision: 0.858 (85.8%)
    Recall: 0.603 (60.3%)

TESTING THRESHOLD = 0.6


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 199
Avg confidence: 0.868
Confidence range: [0.607, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 199
  Manual events: 300
  True Positives: 182
  False Positives: 17
  False Negatives: 118

Metrics:
  Precision: 0.915 (91.5%)
  Recall: 0.607 (60.7%)
  F1 Score: 0.729

By Event Type:

  Entry:
    Manual: 149
    TP: 97, FP: 7, FN: 52
    Precision: 0.933 (93.3%)
    Recall: 0.651 (65.1%)

  Exit:
    Manual: 151
    TP: 85, FP: 10, FN: 66
    Precision: 0.895 (89.5%)
    Recall: 0.563 (56.3%)

THRESHOLD COMPARISON

Threshold    Precision    Recall       F1           Detected     TP       FP       FN      
----------------------------------------------------------------------------------------------------
0.3          0.813        0.797        0.805        294          239      55       61      
0.4          0.

In [3]:
"""ML-First Event Detection Evaluation
Evaluate ML-based event detection against manual ground truth.
No heuristics - pure ML filtering.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys

# Add to path
sys.path.insert(0, '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6')

from beemonitor.processing import EventProcessor
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO

print("="*70)
print("ML-FIRST EVENT DETECTION EVALUATION")
print("="*70)

# Setup
config = Config.default()
#config.models.event_classifier = 'event_classifier_ml_first.pkl'  # Use ML-First model

nest_model = YOLO(config.models.nest_detection)
detector = NestDetector(nest_model, config)

# Paths
input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
#output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/CVPR_Output"
output_folder = "/Users/edwardamoah/Downloads/CVPR_Evaluation_Video_Data_output"
manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'

# Load videos and tracking data
files = [os.path.join(input_data, f) for f in os.listdir(input_data) if 'mp4' in f]
tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.endswith('_tracking_results.csv')]

tracking_data = {}
for file in tracking_files:
    video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
    tracking_data[video_name] = pd.read_csv(file)

print(f"\nDataset:")
print(f"  Videos: {len(files)}")
print(f"  Tracking files: {len(tracking_data)}")

# Load manual ground truth
manual_df = pd.read_csv(manual_csv)
manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()

def parse_manual_time(video, time_str):
    date_part = video.split('_')[1]
    return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")

manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)

print(f"  Manual events: {len(manual_df)}")

def reconstruct_motion_data(tracking_df):
    """Convert tracking CSV to motion_data format."""
    trajectories = []
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        centroids = []
        for _, row in track_data.iterrows():
            centroid_x = (row['x1'] + row['x2']) / 2
            centroid_y = (row['y1'] + row['y2']) / 2
            centroids.append((centroid_x, centroid_y))
        
        bboxes = list(zip(track_data['x1'], track_data['y1'], 
                         track_data['x2'], track_data['y2']))
        frame_numbers = track_data['frame'].tolist()
        
        trajectory = (track_id, centroids, bboxes, frame_numbers)
        trajectories.append(trajectory)
    
    return pd.DataFrame({'tracks': [trajectories]})

# ======================================================================
# PROCESS ALL VIDEOS WITH ML-FIRST
# ======================================================================

print("\n" + "="*70)
print("PROCESSING VIDEOS WITH ML-FIRST DETECTION")
print("="*70)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]
results_by_threshold = {}

for threshold in thresholds:
    print(f"\n{'='*70}")
    print(f"TESTING THRESHOLD = {threshold}")
    print(f"{'='*70}")
    
    processor = EventProcessor(config)
    all_events = []
    per_video_stats = []
    
    for video_name, tracking_df in tracking_data.items():
        # Get video file
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            continue
        video_file = video_file[0]
        
        # Get nests
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  ⚠️  {video_name}: Nest detection failed")
            continue
        
        # Process with ML-First
        motion_data = reconstruct_motion_data(tracking_df)
        
        events = processor.process_tracks(
            motion_data=motion_data,
            nests=nests,
            ml_threshold=threshold
        )
        
        events['video'] = video_name
        all_events.append(events)
        
        # Get manual events for this video
        manual_for_video = len(manual_df[manual_df['video'] == video_name])
        
        per_video_stats.append({
            'video': video_name,
            'detected': len(events),
            'manual': manual_for_video
        })
    
    # Combine all events
    events_df = pd.concat(all_events, ignore_index=True)
    
    print(f"\nTotal events detected: {len(events_df)}")
    print(f"Avg confidence: {events_df['ml_confidence'].mean():.3f}")
    print(f"Confidence range: [{events_df['ml_confidence'].min():.3f}, {events_df['ml_confidence'].max():.3f}]")
    
    # Match to manual ground truth
    def match_events(predicted_events, manual_events, tolerance_sec=3.0):
        """Match predicted events to manual events."""
        matches = []
        used_predicted = set()
        
        for _, man_event in manual_events.iterrows():
            video_name = man_event['video']
            pred_for_video = predicted_events[predicted_events['video'] == video_name]
            
            # Extract video start time
            parts = video_name.split('_')
            date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
            video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            
            for pred_idx, pred_event in pred_for_video.iterrows():
                if pred_idx in used_predicted:
                    continue
                
                # Match action and nest
                pred_nest = int(pred_event['nest'])
                man_nest = int(man_event['nest'])
                
                if (pred_event['action'] == man_event['action'] and pred_nest == man_nest):
                    # Check time difference
                    pred_time = video_start + timedelta(seconds=pred_event['frame_number']/30.0)
                    time_diff = abs(pred_time - man_event['dt'])
                    
                    if time_diff <= timedelta(seconds=tolerance_sec):
                        matches.append({
                            'manual_idx': man_event.name,
                            'predicted_idx': pred_idx,
                            'time_diff': time_diff.total_seconds(),
                            'video': video_name,
                            'action': man_event['action']
                        })
                        used_predicted.add(pred_idx)
                        break
        
        true_positives = len(matches)
        false_positives = len(predicted_events) - true_positives
        false_negatives = len(manual_events) - true_positives
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'tp': true_positives,
            'fp': false_positives,
            'fn': false_negatives,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'matches': matches
        }
    
    metrics = match_events(events_df, manual_df)
    
    # Error analysis by event type
    matched_pred_idx = set([m['predicted_idx'] for m in metrics['matches']])
    matched_manual_idx = set([m['manual_idx'] for m in metrics['matches']])
    
    tp_by_action = {}
    fp_by_action = {}
    fn_by_action = {}
    
    for action in ['Entry', 'Exit']:
        manual_action = manual_df[manual_df['action'] == action]
        tp = sum(1 for m in metrics['matches'] if m['action'] == action)
        fn = len(manual_action) - tp
        
        pred_action = events_df[events_df['action'] == action]
        fp = len(pred_action) - tp
        
        tp_by_action[action] = tp
        fp_by_action[action] = fp
        fn_by_action[action] = fn
    
    # Display results
    print("\n" + "-"*70)
    print("RESULTS")
    print("-"*70)
    
    print(f"\nOverall Performance:")
    print(f"  Predicted events: {len(events_df)}")
    print(f"  Manual events: {len(manual_df)}")
    print(f"  True Positives: {metrics['tp']}")
    print(f"  False Positives: {metrics['fp']}")
    print(f"  False Negatives: {metrics['fn']}")
    
    print(f"\nMetrics:")
    print(f"  Precision: {metrics['precision']:.3f} ({metrics['precision']*100:.1f}%)")
    print(f"  Recall: {metrics['recall']:.3f} ({metrics['recall']*100:.1f}%)")
    print(f"  F1 Score: {metrics['f1']:.3f}")
    
    print(f"\nBy Event Type:")
    for action in ['Entry', 'Exit']:
        manual_count = len(manual_df[manual_df['action'] == action])
        tp = tp_by_action[action]
        fp = fp_by_action[action]
        fn = fn_by_action[action]
        
        recall = tp / manual_count if manual_count > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  {action}:")
        print(f"    Manual: {manual_count}")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")
        print(f"    Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"    Recall: {recall:.3f} ({recall*100:.1f}%)")
    
    # Save results for this threshold
    results_by_threshold[threshold] = {
        'metrics': metrics,
        'events_df': events_df,
        'per_video': pd.DataFrame(per_video_stats)
    }

# ======================================================================
# THRESHOLD COMPARISON
# ======================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON")
print("="*70)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Detected':<12} {'TP':<8} {'FP':<8} {'FN':<8}")
print("-" * 100)

best_f1 = 0
best_threshold = 0.3

for threshold in thresholds:
    m = results_by_threshold[threshold]['metrics']
    detected = len(results_by_threshold[threshold]['events_df'])
    
    print(f"{threshold:<12.1f} {m['precision']:<12.3f} {m['recall']:<12.3f} {m['f1']:<12.3f} {detected:<12} {m['tp']:<8} {m['fp']:<8} {m['fn']:<8}")
    
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_threshold = threshold

print(f"\n✓ Best threshold: {best_threshold} (F1={best_f1:.3f})")

# ======================================================================
# DETAILED RESULTS FOR BEST THRESHOLD
# ======================================================================

print("\n" + "="*70)
print(f"DETAILED RESULTS (Threshold = {best_threshold})")
print("="*70)

best_results = results_by_threshold[best_threshold]
best_metrics = best_results['metrics']
best_events = best_results['events_df']

print(f"\nPer-Video Breakdown:")
print(best_results['per_video'].to_string(index=False))

# Save to CSV
best_results['per_video'].to_csv('ml_first_evaluation_per_video.csv', index=False)
print(f"\n✓ Per-video results saved to ml_first_evaluation_per_video.csv")

# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nML-First Detection:")
print(f"  Window size: 1 frame")
print(f"  Padding: 40 pixels")
print(f"  Minimal trajectory filtering")

print(f"\nBest Configuration:")
print(f"  Threshold: {best_threshold}")
print(f"  Precision: {best_metrics['precision']:.3f} ({best_metrics['precision']*100:.1f}%)")
print(f"  Recall: {best_metrics['recall']:.3f} ({best_metrics['recall']*100:.1f}%)")
print(f"  F1 Score: {best_metrics['f1']:.3f}")

print(f"\nError Analysis:")
print(f"  True Positives: {best_metrics['tp']} (correctly detected)")
print(f"  False Positives: {best_metrics['fp']} (noise detected as events)")
print(f"  False Negatives: {best_metrics['fn']} (missed real events)")

print(f"\nFor CVPR Paper:")
print(f"  \"Our ML-First approach achieves {best_metrics['precision']*100:.1f}% precision")
print(f"   and {best_metrics['recall']*100:.1f}% recall (F1={best_metrics['f1']:.3f})\"")
print(f"  \"No manual parameter tuning - data-driven event detection\"")

print("\n" + "="*70)

ML-FIRST EVENT DETECTION EVALUATION

Dataset:
  Videos: 11
  Tracking files: 11
  Manual events: 300

PROCESSING VIDEOS WITH ML-FIRST DETECTION

TESTING THRESHOLD = 0.3


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 296
Avg confidence: 0.728
Confidence range: [0.302, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 296
  Manual events: 300
  True Positives: 247
  False Positives: 49
  False Negatives: 53

Metrics:
  Precision: 0.834 (83.4%)
  Recall: 0.823 (82.3%)
  F1 Score: 0.829

By Event Type:

  Entry:
    Manual: 149
    TP: 128, FP: 21, FN: 21
    Precision: 0.859 (85.9%)
    Recall: 0.859 (85.9%)

  Exit:
    Manual: 151
    TP: 119, FP: 28, FN: 32
    Precision: 0.810 (81.0%)
    Recall: 0.788 (78.8%)

TESTING THRESHOLD = 0.4


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 257
Avg confidence: 0.786
Confidence range: [0.404, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 257
  Manual events: 300
  True Positives: 221
  False Positives: 36
  False Negatives: 79

Metrics:
  Precision: 0.860 (86.0%)
  Recall: 0.737 (73.7%)
  F1 Score: 0.794

By Event Type:

  Entry:
    Manual: 149
    TP: 116, FP: 16, FN: 33
    Precision: 0.879 (87.9%)
    Recall: 0.779 (77.9%)

  Exit:
    Manual: 151
    TP: 105, FP: 20, FN: 46
    Precision: 0.840 (84.0%)
    Recall: 0.695 (69.5%)

TESTING THRESHOLD = 0.5


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 225
Avg confidence: 0.835
Confidence range: [0.501, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 225
  Manual events: 300
  True Positives: 200
  False Positives: 25
  False Negatives: 100

Metrics:
  Precision: 0.889 (88.9%)
  Recall: 0.667 (66.7%)
  F1 Score: 0.762

By Event Type:

  Entry:
    Manual: 149
    TP: 103, FP: 10, FN: 46
    Precision: 0.912 (91.2%)
    Recall: 0.691 (69.1%)

  Exit:
    Manual: 151
    TP: 97, FP: 15, FN: 54
    Precision: 0.866 (86.6%)
    Recall: 0.642 (64.2%)

TESTING THRESHOLD = 0.6


Quality check failed: Row 4 has inconsistent spacing. Gap 8: 0.0, Average: 51.7
Attempt 1: Quality check failed



Total events detected: 202
Avg confidence: 0.868
Confidence range: [0.611, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 202
  Manual events: 300
  True Positives: 188
  False Positives: 14
  False Negatives: 112

Metrics:
  Precision: 0.931 (93.1%)
  Recall: 0.627 (62.7%)
  F1 Score: 0.749

By Event Type:

  Entry:
    Manual: 149
    TP: 97, FP: 4, FN: 52
    Precision: 0.960 (96.0%)
    Recall: 0.651 (65.1%)

  Exit:
    Manual: 151
    TP: 91, FP: 10, FN: 60
    Precision: 0.901 (90.1%)
    Recall: 0.603 (60.3%)

THRESHOLD COMPARISON

Threshold    Precision    Recall       F1           Detected     TP       FP       FN      
----------------------------------------------------------------------------------------------------
0.3          0.834        0.823        0.829        296          247      49       53      
0.4          0.

In [5]:
"""ML-First Event Detection Evaluation
Evaluate ML-based event detection against manual ground truth.
No heuristics - pure ML filtering.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys

# Add to path
sys.path.insert(0, '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6')

from beemonitor.processing import EventProcessor
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO

print("="*70)
print("ML-FIRST EVENT DETECTION EVALUATION")
print("="*70)

# Setup
config = Config.default()
#config.models.event_classifier = 'event_classifier_ml_first.pkl'  # Use ML-First model

nest_model = YOLO(config.models.nest_detection)
detector = NestDetector(nest_model, config)

# Paths
input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
#output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/CVPR_Output"
output_folder = "/Users/edwardamoah/Downloads/CVPR_Evaluation_Video_Data_output"
manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'

# Load videos and tracking data
files = [os.path.join(input_data, f) for f in os.listdir(input_data) if 'mp4' in f]
tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.endswith('_tracking_results.csv')]

tracking_data = {}
for file in tracking_files:
    video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
    tracking_data[video_name] = pd.read_csv(file)

print(f"\nDataset:")
print(f"  Videos: {len(files)}")
print(f"  Tracking files: {len(tracking_data)}")

# Load manual ground truth
manual_df = pd.read_csv(manual_csv)
manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()

def parse_manual_time(video, time_str):
    date_part = video.split('_')[1]
    return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")

manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)

print(f"  Manual events: {len(manual_df)}")

def reconstruct_motion_data(tracking_df):
    """Convert tracking CSV to motion_data format."""
    trajectories = []
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        centroids = []
        for _, row in track_data.iterrows():
            centroid_x = (row['x1'] + row['x2']) / 2
            centroid_y = (row['y1'] + row['y2']) / 2
            centroids.append((centroid_x, centroid_y))
        
        bboxes = list(zip(track_data['x1'], track_data['y1'], 
                         track_data['x2'], track_data['y2']))
        frame_numbers = track_data['frame'].tolist()
        
        trajectory = (track_id, centroids, bboxes, frame_numbers)
        trajectories.append(trajectory)
    
    return pd.DataFrame({'tracks': [trajectories]})

# ======================================================================
# PROCESS ALL VIDEOS WITH ML-FIRST
# ======================================================================

print("\n" + "="*70)
print("PROCESSING VIDEOS WITH ML-FIRST DETECTION")
print("="*70)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]
results_by_threshold = {}

for threshold in thresholds:
    print(f"\n{'='*70}")
    print(f"TESTING THRESHOLD = {threshold}")
    print(f"{'='*70}")
    
    processor = EventProcessor(config)
    all_events = []
    per_video_stats = []
    
    for video_name, tracking_df in tracking_data.items():
        # Get video file
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            continue
        video_file = video_file[0]
        
        # Get nests
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  ⚠️  {video_name}: Nest detection failed")
            continue
        
        # Process with ML-First
        motion_data = reconstruct_motion_data(tracking_df)
        
        events = processor.process_tracks(
            motion_data=motion_data,
            nests=nests,
            ml_threshold=threshold
        )
        
        events['video'] = video_name
        all_events.append(events)
        
        # Get manual events for this video
        manual_for_video = len(manual_df[manual_df['video'] == video_name])
        
        per_video_stats.append({
            'video': video_name,
            'detected': len(events),
            'manual': manual_for_video
        })
    
    # Combine all events
    events_df = pd.concat(all_events, ignore_index=True)
    
    print(f"\nTotal events detected: {len(events_df)}")
    print(f"Avg confidence: {events_df['ml_confidence'].mean():.3f}")
    print(f"Confidence range: [{events_df['ml_confidence'].min():.3f}, {events_df['ml_confidence'].max():.3f}]")
    
    # Match to manual ground truth
    def match_events(predicted_events, manual_events, tolerance_sec=3.0):
        """Match predicted events to manual events."""
        matches = []
        used_predicted = set()
        
        for _, man_event in manual_events.iterrows():
            video_name = man_event['video']
            pred_for_video = predicted_events[predicted_events['video'] == video_name]
            
            # Extract video start time
            parts = video_name.split('_')
            date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
            video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            
            for pred_idx, pred_event in pred_for_video.iterrows():
                if pred_idx in used_predicted:
                    continue
                
                # Match action and nest
                pred_nest = int(pred_event['nest'])
                man_nest = int(man_event['nest'])
                
                if (pred_event['action'] == man_event['action'] and pred_nest == man_nest):
                    # Check time difference
                    pred_time = video_start + timedelta(seconds=pred_event['frame_number']/30.0)
                    time_diff = abs(pred_time - man_event['dt'])
                    
                    if time_diff <= timedelta(seconds=tolerance_sec):
                        matches.append({
                            'manual_idx': man_event.name,
                            'predicted_idx': pred_idx,
                            'time_diff': time_diff.total_seconds(),
                            'video': video_name,
                            'action': man_event['action']
                        })
                        used_predicted.add(pred_idx)
                        break
        
        true_positives = len(matches)
        false_positives = len(predicted_events) - true_positives
        false_negatives = len(manual_events) - true_positives
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'tp': true_positives,
            'fp': false_positives,
            'fn': false_negatives,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'matches': matches
        }
    
    metrics = match_events(events_df, manual_df)
    
    # Error analysis by event type
    matched_pred_idx = set([m['predicted_idx'] for m in metrics['matches']])
    matched_manual_idx = set([m['manual_idx'] for m in metrics['matches']])
    
    tp_by_action = {}
    fp_by_action = {}
    fn_by_action = {}
    
    for action in ['Entry', 'Exit']:
        manual_action = manual_df[manual_df['action'] == action]
        tp = sum(1 for m in metrics['matches'] if m['action'] == action)
        fn = len(manual_action) - tp
        
        pred_action = events_df[events_df['action'] == action]
        fp = len(pred_action) - tp
        
        tp_by_action[action] = tp
        fp_by_action[action] = fp
        fn_by_action[action] = fn
    
    # Display results
    print("\n" + "-"*70)
    print("RESULTS")
    print("-"*70)
    
    print(f"\nOverall Performance:")
    print(f"  Predicted events: {len(events_df)}")
    print(f"  Manual events: {len(manual_df)}")
    print(f"  True Positives: {metrics['tp']}")
    print(f"  False Positives: {metrics['fp']}")
    print(f"  False Negatives: {metrics['fn']}")
    
    print(f"\nMetrics:")
    print(f"  Precision: {metrics['precision']:.3f} ({metrics['precision']*100:.1f}%)")
    print(f"  Recall: {metrics['recall']:.3f} ({metrics['recall']*100:.1f}%)")
    print(f"  F1 Score: {metrics['f1']:.3f}")
    
    print(f"\nBy Event Type:")
    for action in ['Entry', 'Exit']:
        manual_count = len(manual_df[manual_df['action'] == action])
        tp = tp_by_action[action]
        fp = fp_by_action[action]
        fn = fn_by_action[action]
        
        recall = tp / manual_count if manual_count > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  {action}:")
        print(f"    Manual: {manual_count}")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")
        print(f"    Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"    Recall: {recall:.3f} ({recall*100:.1f}%)")
    
    # Save results for this threshold
    results_by_threshold[threshold] = {
        'metrics': metrics,
        'events_df': events_df,
        'per_video': pd.DataFrame(per_video_stats)
    }

# ======================================================================
# THRESHOLD COMPARISON
# ======================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON")
print("="*70)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Detected':<12} {'TP':<8} {'FP':<8} {'FN':<8}")
print("-" * 100)

best_f1 = 0
best_threshold = 0.3

for threshold in thresholds:
    m = results_by_threshold[threshold]['metrics']
    detected = len(results_by_threshold[threshold]['events_df'])
    
    print(f"{threshold:<12.1f} {m['precision']:<12.3f} {m['recall']:<12.3f} {m['f1']:<12.3f} {detected:<12} {m['tp']:<8} {m['fp']:<8} {m['fn']:<8}")
    
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_threshold = threshold

print(f"\n✓ Best threshold: {best_threshold} (F1={best_f1:.3f})")

# ======================================================================
# DETAILED RESULTS FOR BEST THRESHOLD
# ======================================================================

print("\n" + "="*70)
print(f"DETAILED RESULTS (Threshold = {best_threshold})")
print("="*70)

best_results = results_by_threshold[best_threshold]
best_metrics = best_results['metrics']
best_events = best_results['events_df']

print(f"\nPer-Video Breakdown:")
print(best_results['per_video'].to_string(index=False))

# Save to CSV
best_results['per_video'].to_csv('ml_first_evaluation_per_video.csv', index=False)
print(f"\n✓ Per-video results saved to ml_first_evaluation_per_video.csv")

# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nML-First Detection:")
print(f"  Window size: 1 frame")
print(f"  Padding: 40 pixels")
print(f"  Minimal trajectory filtering")

print(f"\nBest Configuration:")
print(f"  Threshold: {best_threshold}")
print(f"  Precision: {best_metrics['precision']:.3f} ({best_metrics['precision']*100:.1f}%)")
print(f"  Recall: {best_metrics['recall']:.3f} ({best_metrics['recall']*100:.1f}%)")
print(f"  F1 Score: {best_metrics['f1']:.3f}")

print(f"\nError Analysis:")
print(f"  True Positives: {best_metrics['tp']} (correctly detected)")
print(f"  False Positives: {best_metrics['fp']} (noise detected as events)")
print(f"  False Negatives: {best_metrics['fn']} (missed real events)")

print(f"\nFor CVPR Paper:")
print(f"  \"Our ML-First approach achieves {best_metrics['precision']*100:.1f}% precision")
print(f"   and {best_metrics['recall']*100:.1f}% recall (F1={best_metrics['f1']:.3f})\"")
print(f"  \"No manual parameter tuning - data-driven event detection\"")

print("\n" + "="*70)






# ✓ Selected folder: CVPR_Evaluation_Video_Data
#   Found 11 video files

# ==================================================
# BATCH ANALYSIS STARTED
# ==================================================
# Videos: 11
# Workers: 4
# Starting batch analysis of folder: CVPR_Evaluation_Video_Data
# Processing 11 videos with 4 parallel workers
# Started at: 15:05:34
# ✓ Interaction metrics enabled (proximity=50px)
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4 (+1 more)
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_00_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_10_01.mp4 in 11m 48s (1/11)
# ⚙️  Processing: mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_00_00.mp4, mendels_2024-04-30_09_20_00.mp4 (+1 more)
# ✓ Completed mendels_2024-05-08_15_00_00.mp4 in 24m 25s (2/11)
# ⚙️  Processing: mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_20_00.mp4, mendels_2024-05-08_15_30_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_00_00.mp4 in 31m 48s (3/11)
# ⚙️  Processing: mendels_2024-04-30_09_20_00.mp4, mendels_2024-05-08_15_30_00.mp4, mendels_2024-04-30_09_30_00.mp4 (+1 more)
# ✓ Completed mendels_2024-05-23_12_40_00.mp4 in 36m 5s (4/11)
# ⚙️  Processing: mendels_2024-05-08_15_30_00.mp4, mendels_2024-04-30_09_30_00.mp4, mendels_2024-05-23_12_00_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_20_00.mp4 in 1h 1m 15s (5/11)
# ⚙️  Processing: mendels_2024-05-08_15_30_00.mp4, mendels_2024-04-30_09_30_00.mp4, mendels_2024-05-08_15_50_00.mp4 (+1 more)
# ✓ Completed mendels_2024-05-23_12_00_00.mp4 in 37m 13s (6/11)
# ⚙️  Processing: mendels_2024-04-30_09_30_00.mp4, mendels_2024-05-08_15_50_00.mp4, mendels_2024-04-30_09_40_01.mp4 (+1 more)
# ✓ Completed mendels_2024-05-08_15_30_00.mp4 in 50m 24s (7/11)
# ✓ Completed mendels_2024-04-30_09_30_00.mp4 in 48m 23s (8/11)
# ✓ Completed mendels_2024-04-30_09_40_01.mp4 in 7m 42s (9/11)
# ✓ Completed mendels_2024-05-08_15_50_00.mp4 in 8m 9s (10/11)
# ✓ Completed mendels_2024-05-23_18_20_01.mp4 in 10m 42s (11/11)

# ==================================================
# ✓ BATCH ANALYSIS COMPLETE!
# ==================================================
# Videos processed: 11/11
# Total time: 1h 25m 32s
# Average per video: 7m 46s
# Finished at: 16:31:06

# ==================================================
# BATCH ANALYSIS COMPLETE
# ==================================================
# Total videos: 11
# Successful: 11
# Failed: 0
# Total events: 274

ML-FIRST EVENT DETECTION EVALUATION

Dataset:
  Videos: 11
  Tracking files: 11
  Manual events: 300

PROCESSING VIDEOS WITH ML-FIRST DETECTION

TESTING THRESHOLD = 0.3

Total events detected: 330
Avg confidence: 0.810
Confidence range: [0.301, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 330
  Manual events: 300
  True Positives: 266
  False Positives: 64
  False Negatives: 34

Metrics:
  Precision: 0.806 (80.6%)
  Recall: 0.887 (88.7%)
  F1 Score: 0.844

By Event Type:

  Entry:
    Manual: 149
    TP: 133, FP: 31, FN: 16
    Precision: 0.811 (81.1%)
    Recall: 0.893 (89.3%)

  Exit:
    Manual: 151
    TP: 133, FP: 33, FN: 18
    Precision: 0.801 (80.1%)
    Recall: 0.881 (88.1%)

TESTING THRESHOLD = 0.4

Total events detected: 312
Avg confidence: 0.837
Confidence range: [0.400, 1.000]

-------------------------------------------

In [10]:
"""ML-First Event Detection Evaluation
Evaluate ML-based event detection against manual ground truth.
No heuristics - pure ML filtering.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys

# Add to path
sys.path.insert(0, '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6')

from beemonitor.processing import EventProcessor
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO

print("="*70)
print("ML-FIRST EVENT DETECTION EVALUATION")
print("="*70)

# Setup
config = Config.default()
#config.models.event_classifier = 'event_classifier_ml_first.pkl'  # Use ML-First model

nest_model = YOLO(config.models.nest_detection)
detector = NestDetector(nest_model, config)

# Paths
input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
#output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/CVPR_Output"
output_folder = "/Users/edwardamoah/Downloads/CVPR_Evaluation_Video_Data_output"
manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'

# Load videos and tracking data
files = [os.path.join(input_data, f) for f in os.listdir(input_data) if 'mp4' in f]
tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.endswith('_tracking_results.csv')]

tracking_data = {}
for file in tracking_files:
    video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
    tracking_data[video_name] = pd.read_csv(file)

print(f"\nDataset:")
print(f"  Videos: {len(files)}")
print(f"  Tracking files: {len(tracking_data)}")

# Load manual ground truth
manual_df = pd.read_csv(manual_csv)
manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()

def parse_manual_time(video, time_str):
    date_part = video.split('_')[1]
    return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")

manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)

print(f"  Manual events: {len(manual_df)}")

def reconstruct_motion_data(tracking_df):
    """Convert tracking CSV to motion_data format."""
    trajectories = []
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        centroids = []
        for _, row in track_data.iterrows():
            centroid_x = (row['x1'] + row['x2']) / 2
            centroid_y = (row['y1'] + row['y2']) / 2
            centroids.append((centroid_x, centroid_y))
        
        bboxes = list(zip(track_data['x1'], track_data['y1'], 
                         track_data['x2'], track_data['y2']))
        frame_numbers = track_data['frame'].tolist()
        
        trajectory = (track_id, centroids, bboxes, frame_numbers)
        trajectories.append(trajectory)
    
    return pd.DataFrame({'tracks': [trajectories]})

# ======================================================================
# PROCESS ALL VIDEOS WITH ML-FIRST
# ======================================================================

print("\n" + "="*70)
print("PROCESSING VIDEOS WITH ML-FIRST DETECTION")
print("="*70)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]
results_by_threshold = {}

for threshold in thresholds:
    print(f"\n{'='*70}")
    print(f"TESTING THRESHOLD = {threshold}")
    print(f"{'='*70}")
    
    processor = EventProcessor(config)
    all_events = []
    per_video_stats = []
    
    for video_name, tracking_df in tracking_data.items():
        # Get video file
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            continue
        video_file = video_file[0]
        
        # Get nests
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  ⚠️  {video_name}: Nest detection failed")
            continue
        
        # Process with ML-First
        motion_data = reconstruct_motion_data(tracking_df)
        
        events = processor.process_tracks(
            motion_data=motion_data,
            nests=nests,
            ml_threshold=threshold
        )
        
        events['video'] = video_name
        all_events.append(events)
        
        # Get manual events for this video
        manual_for_video = len(manual_df[manual_df['video'] == video_name])
        
        per_video_stats.append({
            'video': video_name,
            'detected': len(events),
            'manual': manual_for_video
        })
    
    # Combine all events
    events_df = pd.concat(all_events, ignore_index=True)
    
    print(f"\nTotal events detected: {len(events_df)}")
    print(f"Avg confidence: {events_df['ml_confidence'].mean():.3f}")
    print(f"Confidence range: [{events_df['ml_confidence'].min():.3f}, {events_df['ml_confidence'].max():.3f}]")
    
    # Match to manual ground truth
    def match_events(predicted_events, manual_events, tolerance_sec=3.0):
        """Match predicted events to manual events."""
        matches = []
        used_predicted = set()
        
        for _, man_event in manual_events.iterrows():
            video_name = man_event['video']
            pred_for_video = predicted_events[predicted_events['video'] == video_name]
            
            # Extract video start time
            parts = video_name.split('_')
            date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
            video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            
            for pred_idx, pred_event in pred_for_video.iterrows():
                if pred_idx in used_predicted:
                    continue
                
                # Match action and nest
                pred_nest = int(pred_event['nest'])
                man_nest = int(man_event['nest'])
                
                if (pred_event['action'] == man_event['action'] and pred_nest == man_nest):
                    # Check time difference
                    pred_time = video_start + timedelta(seconds=pred_event['frame_number']/30.0)
                    time_diff = abs(pred_time - man_event['dt'])
                    
                    if time_diff <= timedelta(seconds=tolerance_sec):
                        matches.append({
                            'manual_idx': man_event.name,
                            'predicted_idx': pred_idx,
                            'time_diff': time_diff.total_seconds(),
                            'video': video_name,
                            'action': man_event['action']
                        })
                        used_predicted.add(pred_idx)
                        break
        
        true_positives = len(matches)
        false_positives = len(predicted_events) - true_positives
        false_negatives = len(manual_events) - true_positives
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'tp': true_positives,
            'fp': false_positives,
            'fn': false_negatives,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'matches': matches
        }
    
    metrics = match_events(events_df, manual_df)
    
    # Error analysis by event type
    matched_pred_idx = set([m['predicted_idx'] for m in metrics['matches']])
    matched_manual_idx = set([m['manual_idx'] for m in metrics['matches']])
    
    tp_by_action = {}
    fp_by_action = {}
    fn_by_action = {}
    
    for action in ['Entry', 'Exit']:
        manual_action = manual_df[manual_df['action'] == action]
        tp = sum(1 for m in metrics['matches'] if m['action'] == action)
        fn = len(manual_action) - tp
        
        pred_action = events_df[events_df['action'] == action]
        fp = len(pred_action) - tp
        
        tp_by_action[action] = tp
        fp_by_action[action] = fp
        fn_by_action[action] = fn
    
    # Display results
    print("\n" + "-"*70)
    print("RESULTS")
    print("-"*70)
    
    print(f"\nOverall Performance:")
    print(f"  Predicted events: {len(events_df)}")
    print(f"  Manual events: {len(manual_df)}")
    print(f"  True Positives: {metrics['tp']}")
    print(f"  False Positives: {metrics['fp']}")
    print(f"  False Negatives: {metrics['fn']}")
    
    print(f"\nMetrics:")
    print(f"  Precision: {metrics['precision']:.3f} ({metrics['precision']*100:.1f}%)")
    print(f"  Recall: {metrics['recall']:.3f} ({metrics['recall']*100:.1f}%)")
    print(f"  F1 Score: {metrics['f1']:.3f}")
    
    print(f"\nBy Event Type:")
    for action in ['Entry', 'Exit']:
        manual_count = len(manual_df[manual_df['action'] == action])
        tp = tp_by_action[action]
        fp = fp_by_action[action]
        fn = fn_by_action[action]
        
        recall = tp / manual_count if manual_count > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  {action}:")
        print(f"    Manual: {manual_count}")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")
        print(f"    Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"    Recall: {recall:.3f} ({recall*100:.1f}%)")
    
    # Save results for this threshold
    results_by_threshold[threshold] = {
        'metrics': metrics,
        'events_df': events_df,
        'per_video': pd.DataFrame(per_video_stats)
    }

# ======================================================================
# THRESHOLD COMPARISON
# ======================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON")
print("="*70)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Detected':<12} {'TP':<8} {'FP':<8} {'FN':<8}")
print("-" * 100)

best_f1 = 0
best_threshold = 0.3

for threshold in thresholds:
    m = results_by_threshold[threshold]['metrics']
    detected = len(results_by_threshold[threshold]['events_df'])
    
    print(f"{threshold:<12.1f} {m['precision']:<12.3f} {m['recall']:<12.3f} {m['f1']:<12.3f} {detected:<12} {m['tp']:<8} {m['fp']:<8} {m['fn']:<8}")
    
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_threshold = threshold

print(f"\n✓ Best threshold: {best_threshold} (F1={best_f1:.3f})")

# ======================================================================
# DETAILED RESULTS FOR BEST THRESHOLD
# ======================================================================

print("\n" + "="*70)
print(f"DETAILED RESULTS (Threshold = {best_threshold})")
print("="*70)

best_results = results_by_threshold[best_threshold]
best_metrics = best_results['metrics']
best_events = best_results['events_df']

print(f"\nPer-Video Breakdown:")
print(best_results['per_video'].to_string(index=False))

# Save to CSV
best_results['per_video'].to_csv('ml_first_evaluation_per_video.csv', index=False)
print(f"\n✓ Per-video results saved to ml_first_evaluation_per_video.csv")

# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nML-First Detection:")
print(f"  Window size: 1 frame")
print(f"  Padding: 40 pixels")
print(f"  Minimal trajectory filtering")

print(f"\nBest Configuration:")
print(f"  Threshold: {best_threshold}")
print(f"  Precision: {best_metrics['precision']:.3f} ({best_metrics['precision']*100:.1f}%)")
print(f"  Recall: {best_metrics['recall']:.3f} ({best_metrics['recall']*100:.1f}%)")
print(f"  F1 Score: {best_metrics['f1']:.3f}")

print(f"\nError Analysis:")
print(f"  True Positives: {best_metrics['tp']} (correctly detected)")
print(f"  False Positives: {best_metrics['fp']} (noise detected as events)")
print(f"  False Negatives: {best_metrics['fn']} (missed real events)")

print(f"\nFor CVPR Paper:")
print(f"  \"Our ML-First approach achieves {best_metrics['precision']*100:.1f}% precision")
print(f"   and {best_metrics['recall']*100:.1f}% recall (F1={best_metrics['f1']:.3f})\"")
print(f"  \"No manual parameter tuning - data-driven event detection\"")

print("\n" + "="*70)


# ✓ Selected folder: CVPR_Evaluation_Video_Data
#   Found 11 video files

# ==================================================
# BATCH ANALYSIS STARTED
# ==================================================
# Videos: 11
# Workers: 4
# Starting batch analysis of folder: CVPR_Evaluation_Video_Data
# Processing 11 videos with 4 parallel workers
# Started at: 20:21:22
# ✓ Interaction metrics enabled (proximity=50px)
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4
# ⚙️  Processing: mendels_2024-05-08_15_00_00.mp4, mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4 (+1 more)
# ⚙️  Processing: mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_00_00.mp4 (+1 more)
# ✓ Completed mendels_2024-05-08_15_00_00.mp4 in 1h 29m 37s (1/11)
# ⚙️  Processing: mendels_2024-04-30_09_10_01.mp4, mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_20_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_00_00.mp4 in 1h 32m 11s (2/11)
# ⚙️  Processing: mendels_2024-05-23_12_40_00.mp4, mendels_2024-04-30_09_20_00.mp4, mendels_2024-05-08_15_30_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_10_01.mp4 in 1h 33m 3s (3/11)
# ⚙️  Processing: mendels_2024-04-30_09_20_00.mp4, mendels_2024-05-08_15_30_00.mp4, mendels_2024-04-30_09_30_00.mp4 (+1 more)
# ✓ Completed mendels_2024-05-23_12_40_00.mp4 in 1h 40m 26s (4/11)
# ⚙️  Processing: mendels_2024-05-08_15_30_00.mp4, mendels_2024-04-30_09_30_00.mp4, mendels_2024-05-23_12_00_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_20_00.mp4 in 1h 30m 46s (5/11)
# ⚙️  Processing: mendels_2024-05-08_15_30_00.mp4, mendels_2024-05-23_12_00_00.mp4, mendels_2024-05-08_15_50_00.mp4 (+1 more)
# ✓ Completed mendels_2024-04-30_09_30_00.mp4 in 1h 28m 43s (6/11)
# ⚙️  Processing: mendels_2024-05-23_12_00_00.mp4, mendels_2024-05-08_15_50_00.mp4, mendels_2024-04-30_09_40_01.mp4 (+1 more)
# ✓ Completed mendels_2024-05-08_15_30_00.mp4 in 1h 29m 58s (7/11)
# ✓ Completed mendels_2024-05-23_12_00_00.mp4 in 1h 22m 35s (8/11)
# ✓ Completed mendels_2024-04-30_09_40_01.mp4 in 59m 57s (9/11)
# ✓ Completed mendels_2024-05-08_15_50_00.mp4 in 1h 5m 33s (10/11)
# ✓ Completed mendels_2024-05-23_18_20_01.mp4 in 1h 8m 53s (11/11)

# ==================================================
# ✓ BATCH ANALYSIS COMPLETE!
# ==================================================
# Videos processed: 11/11
# Total time: 4h 11m 4s
# Average per video: 22m 49s
# Finished at: 00:32:26

# ==================================================
# BATCH ANALYSIS COMPLETE
# ==================================================
# Total videos: 11
# Successful: 11
# Failed: 0
# Total events: 281

ML-FIRST EVENT DETECTION EVALUATION

Dataset:
  Videos: 11
  Tracking files: 11
  Manual events: 300

PROCESSING VIDEOS WITH ML-FIRST DETECTION

TESTING THRESHOLD = 0.3

Total events detected: 323
Avg confidence: 0.826
Confidence range: [0.301, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 323
  Manual events: 300
  True Positives: 272
  False Positives: 51
  False Negatives: 28

Metrics:
  Precision: 0.842 (84.2%)
  Recall: 0.907 (90.7%)
  F1 Score: 0.873

By Event Type:

  Entry:
    Manual: 149
    TP: 136, FP: 24, FN: 13
    Precision: 0.850 (85.0%)
    Recall: 0.913 (91.3%)

  Exit:
    Manual: 151
    TP: 136, FP: 27, FN: 15
    Precision: 0.834 (83.4%)
    Recall: 0.901 (90.1%)

TESTING THRESHOLD = 0.4

Total events detected: 308
Avg confidence: 0.849
Confidence range: [0.400, 1.000]

-------------------------------------------

In [23]:
"""ML-First Event Detection Evaluation
Evaluate ML-based event detection against manual ground truth.
No heuristics - pure ML filtering.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys

# Add to path
sys.path.insert(0, '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6')

from beemonitor.processing import EventProcessor
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO

print("="*70)
print("ML-FIRST EVENT DETECTION EVALUATION")
print("="*70)

# Setup
config = Config.default()
#config.models.event_classifier = 'event_classifier_ml_first.pkl'  # Use ML-First model

nest_model = YOLO(config.models.nest_detection)
detector = NestDetector(nest_model, config)

# Paths
input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
#output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/CVPR_Output"
output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/CVPR_Cloud_Output/CVPR_Output"
manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'

# Load videos and tracking data
files = [os.path.join(input_data, f) for f in os.listdir(input_data) if 'mp4' in f]
tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.endswith('_tracking_results.csv')]

tracking_data = {}
for file in tracking_files:
    video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
    tracking_data[video_name] = pd.read_csv(file)

print(f"\nDataset:")
print(f"  Videos: {len(files)}")
print(f"  Tracking files: {len(tracking_data)}")

# Load manual ground truth
manual_df = pd.read_csv(manual_csv)
manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()

def parse_manual_time(video, time_str):
    date_part = video.split('_')[1]
    return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")

manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)

print(f"  Manual events: {len(manual_df)}")

def reconstruct_motion_data(tracking_df):
    """Convert tracking CSV to motion_data format."""
    trajectories = []
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        centroids = []
        for _, row in track_data.iterrows():
            centroid_x = (row['x1'] + row['x2']) / 2
            centroid_y = (row['y1'] + row['y2']) / 2
            centroids.append((centroid_x, centroid_y))
        
        bboxes = list(zip(track_data['x1'], track_data['y1'], 
                         track_data['x2'], track_data['y2']))
        frame_numbers = track_data['frame'].tolist()
        
        trajectory = (track_id, centroids, bboxes, frame_numbers)
        trajectories.append(trajectory)
    
    return pd.DataFrame({'tracks': [trajectories]})

# ======================================================================
# PROCESS ALL VIDEOS WITH ML-FIRST
# ======================================================================

print("\n" + "="*70)
print("PROCESSING VIDEOS WITH ML-FIRST DETECTION")
print("="*70)

# Test multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6]
results_by_threshold = {}

for threshold in thresholds:
    print(f"\n{'='*70}")
    print(f"TESTING THRESHOLD = {threshold}")
    print(f"{'='*70}")
    
    processor = EventProcessor(config)
    all_events = []
    per_video_stats = []
    
    for video_name, tracking_df in tracking_data.items():
        # Get video file
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            continue
        video_file = video_file[0]
        
        # Get nests
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  ⚠️  {video_name}: Nest detection failed")
            continue
        
        # Process with ML-First
        motion_data = reconstruct_motion_data(tracking_df)
        
        events = processor.process_tracks(
            motion_data=motion_data,
            nests=nests,
            ml_threshold=threshold
        )
        
        events['video'] = video_name
        all_events.append(events)
        
        # Get manual events for this video
        manual_for_video = len(manual_df[manual_df['video'] == video_name])
        
        per_video_stats.append({
            'video': video_name,
            'detected': len(events),
            'manual': manual_for_video
        })
    
    # Combine all events
    events_df = pd.concat(all_events, ignore_index=True)
    
    print(f"\nTotal events detected: {len(events_df)}")
    print(f"Avg confidence: {events_df['ml_confidence'].mean():.3f}")
    print(f"Confidence range: [{events_df['ml_confidence'].min():.3f}, {events_df['ml_confidence'].max():.3f}]")
    
    # Match to manual ground truth
    def match_events(predicted_events, manual_events, tolerance_sec=3.0):
        """Match predicted events to manual events."""
        matches = []
        used_predicted = set()
        
        for _, man_event in manual_events.iterrows():
            video_name = man_event['video']
            pred_for_video = predicted_events[predicted_events['video'] == video_name]
            
            # Extract video start time
            parts = video_name.split('_')
            date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
            video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
            
            for pred_idx, pred_event in pred_for_video.iterrows():
                if pred_idx in used_predicted:
                    continue
                
                # Match action and nest
                pred_nest = int(pred_event['nest'])
                man_nest = int(man_event['nest'])
                
                if (pred_event['action'] == man_event['action'] and pred_nest == man_nest):
                    # Check time difference
                    pred_time = video_start + timedelta(seconds=pred_event['frame_number']/30.0)
                    time_diff = abs(pred_time - man_event['dt'])
                    
                    if time_diff <= timedelta(seconds=tolerance_sec):
                        matches.append({
                            'manual_idx': man_event.name,
                            'predicted_idx': pred_idx,
                            'time_diff': time_diff.total_seconds(),
                            'video': video_name,
                            'action': man_event['action']
                        })
                        used_predicted.add(pred_idx)
                        break
        
        true_positives = len(matches)
        false_positives = len(predicted_events) - true_positives
        false_negatives = len(manual_events) - true_positives
        
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'tp': true_positives,
            'fp': false_positives,
            'fn': false_negatives,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'matches': matches
        }
    
    metrics = match_events(events_df, manual_df)
    
    # Error analysis by event type
    matched_pred_idx = set([m['predicted_idx'] for m in metrics['matches']])
    matched_manual_idx = set([m['manual_idx'] for m in metrics['matches']])
    
    tp_by_action = {}
    fp_by_action = {}
    fn_by_action = {}
    
    for action in ['Entry', 'Exit']:
        manual_action = manual_df[manual_df['action'] == action]
        tp = sum(1 for m in metrics['matches'] if m['action'] == action)
        fn = len(manual_action) - tp
        
        pred_action = events_df[events_df['action'] == action]
        fp = len(pred_action) - tp
        
        tp_by_action[action] = tp
        fp_by_action[action] = fp
        fn_by_action[action] = fn
    
    # Display results
    print("\n" + "-"*70)
    print("RESULTS")
    print("-"*70)
    
    print(f"\nOverall Performance:")
    print(f"  Predicted events: {len(events_df)}")
    print(f"  Manual events: {len(manual_df)}")
    print(f"  True Positives: {metrics['tp']}")
    print(f"  False Positives: {metrics['fp']}")
    print(f"  False Negatives: {metrics['fn']}")
    
    print(f"\nMetrics:")
    print(f"  Precision: {metrics['precision']:.3f} ({metrics['precision']*100:.1f}%)")
    print(f"  Recall: {metrics['recall']:.3f} ({metrics['recall']*100:.1f}%)")
    print(f"  F1 Score: {metrics['f1']:.3f}")
    
    print(f"\nBy Event Type:")
    for action in ['Entry', 'Exit']:
        manual_count = len(manual_df[manual_df['action'] == action])
        tp = tp_by_action[action]
        fp = fp_by_action[action]
        fn = fn_by_action[action]
        
        recall = tp / manual_count if manual_count > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  {action}:")
        print(f"    Manual: {manual_count}")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}")
        print(f"    Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"    Recall: {recall:.3f} ({recall*100:.1f}%)")
    
    # Save results for this threshold
    results_by_threshold[threshold] = {
        'metrics': metrics,
        'events_df': events_df,
        'per_video': pd.DataFrame(per_video_stats)
    }

# ======================================================================
# THRESHOLD COMPARISON
# ======================================================================

print("\n" + "="*70)
print("THRESHOLD COMPARISON")
print("="*70)

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Detected':<12} {'TP':<8} {'FP':<8} {'FN':<8}")
print("-" * 100)

best_f1 = 0
best_threshold = 0.3

for threshold in thresholds:
    m = results_by_threshold[threshold]['metrics']
    detected = len(results_by_threshold[threshold]['events_df'])
    
    print(f"{threshold:<12.1f} {m['precision']:<12.3f} {m['recall']:<12.3f} {m['f1']:<12.3f} {detected:<12} {m['tp']:<8} {m['fp']:<8} {m['fn']:<8}")
    
    if m['f1'] > best_f1:
        best_f1 = m['f1']
        best_threshold = threshold

print(f"\n✓ Best threshold: {best_threshold} (F1={best_f1:.3f})")

# ======================================================================
# DETAILED RESULTS FOR BEST THRESHOLD
# ======================================================================

print("\n" + "="*70)
print(f"DETAILED RESULTS (Threshold = {best_threshold})")
print("="*70)

best_results = results_by_threshold[best_threshold]
best_metrics = best_results['metrics']
best_events = best_results['events_df']

print(f"\nPer-Video Breakdown:")
print(best_results['per_video'].to_string(index=False))

# Save to CSV
best_results['per_video'].to_csv('ml_first_evaluation_per_video.csv', index=False)
print(f"\n✓ Per-video results saved to ml_first_evaluation_per_video.csv")

# ======================================================================
# SUMMARY
# ======================================================================

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nML-First Detection:")
print(f"  Window size: 1 frame")
print(f"  Padding: 40 pixels")
print(f"  Minimal trajectory filtering")

print(f"\nBest Configuration:")
print(f"  Threshold: {best_threshold}")
print(f"  Precision: {best_metrics['precision']:.3f} ({best_metrics['precision']*100:.1f}%)")
print(f"  Recall: {best_metrics['recall']:.3f} ({best_metrics['recall']*100:.1f}%)")
print(f"  F1 Score: {best_metrics['f1']:.3f}")

print(f"\nError Analysis:")
print(f"  True Positives: {best_metrics['tp']} (correctly detected)")
print(f"  False Positives: {best_metrics['fp']} (noise detected as events)")
print(f"  False Negatives: {best_metrics['fn']} (missed real events)")

print(f"\nFor CVPR Paper:")
print(f"  \"Our ML-First approach achieves {best_metrics['precision']*100:.1f}% precision")
print(f"   and {best_metrics['recall']*100:.1f}% recall (F1={best_metrics['f1']:.3f})\"")
print(f"  \"No manual parameter tuning - data-driven event detection\"")

print("\n" + "="*70)


# CLOUD RUN

ML-FIRST EVENT DETECTION EVALUATION

Dataset:
  Videos: 11
  Tracking files: 11
  Manual events: 300

PROCESSING VIDEOS WITH ML-FIRST DETECTION

TESTING THRESHOLD = 0.3

Total events detected: 312
Avg confidence: 0.818
Confidence range: [0.301, 1.000]

----------------------------------------------------------------------
RESULTS
----------------------------------------------------------------------

Overall Performance:
  Predicted events: 312
  Manual events: 300
  True Positives: 257
  False Positives: 55
  False Negatives: 43

Metrics:
  Precision: 0.824 (82.4%)
  Recall: 0.857 (85.7%)
  F1 Score: 0.840

By Event Type:

  Entry:
    Manual: 149
    TP: 132, FP: 29, FN: 17
    Precision: 0.820 (82.0%)
    Recall: 0.886 (88.6%)

  Exit:
    Manual: 151
    TP: 125, FP: 26, FN: 26
    Precision: 0.828 (82.8%)
    Recall: 0.828 (82.8%)

TESTING THRESHOLD = 0.4

Total events detected: 298
Avg confidence: 0.841
Confidence range: [0.400, 1.000]

-------------------------------------------

In [ ]:
# Inconsistency in cloud run is proabably due to the 1.7.2 version of sklearn, which is 1.8.0 in the cloud (or some other package used for ml-based filetring of tracks)

In [24]:
# (venv) edwardamoah@Edwards-MacBook-Pro-2 BeeMonitor_eai6 % python scripts/cross_validation_classifier.py
# ======================================================================
# LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION
# ML Event Classifier Evaluation
# ======================================================================

# Found 11 tracking files

# ======================================================================
# EXTRACTING FEATURES FROM ALL EVENTS
# ======================================================================

# Processing mendels_2024-05-23_18_20_01...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_18_20_01.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_18_20_01.png
#   Detected 153 candidate events

# Processing mendels_2024-05-08_15_30_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_30_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_30_00.png
#   Detected 62 candidate events

# Processing mendels_2024-04-30_09_30_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_30_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_30_00.png
#   Detected 55 candidate events

# Processing mendels_2024-04-30_09_20_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_20_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_20_00.png
#   Detected 24 candidate events

# Processing mendels_2024-04-30_09_10_01...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_10_01.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_10_01.png
#   Detected 76 candidate events

# Processing mendels_2024-05-08_15_00_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_00_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_00_00.png
#   Detected 81 candidate events

# Processing mendels_2024-04-30_09_00_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_00_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_00_00.png
#   Detected 110 candidate events

# Processing mendels_2024-05-08_15_50_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_50_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-08_15_50_00.png
#   Detected 89 candidate events

# Processing mendels_2024-05-23_12_40_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_40_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_40_00.png
#   Detected 129 candidate events

# Processing mendels_2024-05-23_12_00_00...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_00_00.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_00_00.png
#   Detected 65 candidate events

# Processing mendels_2024-04-30_09_40_01...
# INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
# INFO:beemonitor.detection.nest_detector:Detection attempt 1/10
# INFO:beemonitor.detection.nest_detector:Processing nest detections...
# INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_40_01.png
# INFO:beemonitor.detection.nest_detector:All quality checks passed
# INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
# INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_40_01.png
#   Detected 31 candidate events

# ✓ Extracted features for 875 detected events

# ======================================================================
# MATCHING TO MANUAL LABELS
# ======================================================================

# Labeled 875 events:
#   Real events (TP candidates): 287
#   Noise (FP candidates): 588

# ======================================================================
# LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION
# ======================================================================

# Fold: Testing on mendels_2024-04-30_09_00_00
#   Train: 765 (278 real, 487 noise)
#   Test:  110 (9 real, 101 noise)
#   P: 1.000, R: 0.778, F1: 0.875

# Fold: Testing on mendels_2024-04-30_09_10_01
#   Train: 799 (272 real, 527 noise)
#   Test:  76 (15 real, 61 noise)
#   P: 1.000, R: 0.867, F1: 0.929

# Fold: Testing on mendels_2024-04-30_09_20_00
#   Train: 851 (279 real, 572 noise)
#   Test:  24 (8 real, 16 noise)
#   P: 0.750, R: 0.750, F1: 0.750

# Fold: Testing on mendels_2024-04-30_09_30_00
#   Train: 820 (268 real, 552 noise)
#   Test:  55 (19 real, 36 noise)
#   P: 0.875, R: 0.737, F1: 0.800

# Fold: Testing on mendels_2024-04-30_09_40_01
#   Train: 844 (282 real, 562 noise)
#   Test:  31 (5 real, 26 noise)
#   P: 1.000, R: 1.000, F1: 1.000

# Fold: Testing on mendels_2024-05-08_15_00_00
#   Train: 794 (263 real, 531 noise)
#   Test:  81 (24 real, 57 noise)
#   P: 0.774, R: 1.000, F1: 0.873

# Fold: Testing on mendels_2024-05-08_15_30_00
#   Train: 813 (261 real, 552 noise)
#   Test:  62 (26 real, 36 noise)
#   P: 0.867, R: 1.000, F1: 0.929

# Fold: Testing on mendels_2024-05-08_15_50_00
#   Train: 786 (264 real, 522 noise)
#   Test:  89 (23 real, 66 noise)
#   P: 0.958, R: 1.000, F1: 0.979

# Fold: Testing on mendels_2024-05-23_12_00_00
#   Train: 810 (260 real, 550 noise)
#   Test:  65 (27 real, 38 noise)
#   P: 0.964, R: 1.000, F1: 0.982

# Fold: Testing on mendels_2024-05-23_12_40_00
#   Train: 746 (225 real, 521 noise)
#   Test:  129 (62 real, 67 noise)
#   P: 0.951, R: 0.935, F1: 0.943

# Fold: Testing on mendels_2024-05-23_18_20_01
#   Train: 722 (218 real, 504 noise)
#   Test:  153 (69 real, 84 noise)
#   P: 0.887, R: 0.913, F1: 0.900

# ======================================================================
# CROSS-VALIDATION SUMMARY
# ======================================================================

# POOLED METRICS (aggregated predictions):
#   Precision: 0.905
#   Recall:    0.927
#   F1 Score:  0.916

# MEAN PER-FOLD METRICS (± std):
#   Precision: 0.912 ± 0.085
#   Recall:    0.907 ± 0.103
#   F1 Score:  0.905 ± 0.074

# AGGREGATE CONFUSION MATRIX:
#   TP: 266  FP: 28
#   FN: 21  TN: 560

# PER-VIDEO BREAKDOWN:
# Video                                    P        R        F1       TP    FP    FN
# --------------------------------------------------------------------------------
# mendels_2024-04-30_09_00_00              1.000    0.778    0.875    7     0     2
# mendels_2024-04-30_09_10_01              1.000    0.867    0.929    13    0     2
# mendels_2024-04-30_09_20_00              0.750    0.750    0.750    6     2     2
# mendels_2024-04-30_09_30_00              0.875    0.737    0.800    14    2     5
# mendels_2024-04-30_09_40_01              1.000    1.000    1.000    5     0     0
# mendels_2024-05-08_15_00_00              0.774    1.000    0.873    24    7     0
# mendels_2024-05-08_15_30_00              0.867    1.000    0.929    26    4     0
# mendels_2024-05-08_15_50_00              0.958    1.000    0.979    23    1     0
# mendels_2024-05-23_12_00_00              0.964    1.000    0.982    27    1     0
# mendels_2024-05-23_12_40_00              0.951    0.935    0.943    58    3     4
# mendels_2024-05-23_18_20_01              0.887    0.913    0.900    63    8     6

# ======================================================================
# FOR YOUR RESEARCH PAPER
# ======================================================================

# METHODOLOGY TEXT (Section 2.6):
# "To validate the ML classifier's generalization across different recording 
# conditions, we performed leave-one-video-out cross-validation. For each fold, 
# the classifier was trained on trajectory features from 10 videos and 
# tested on the held-out video, repeated for all 11 videos."

# RESULTS TEXT (New Section 3.5 - ML Classifier Validation):
# "The Random Forest event classifier achieved 90.5% precision and 
# 92.7% recall (F1 = 0.916) using leave-one-video-out 
# cross-validation. Mean per-fold performance was 0.905 ± 0.074 F1, 
# with individual video F1 scores ranging from 0.750 to 1.000."

# LIMITATION TEXT (Section 4.1):
# "The ML event classifier requires training data from annotated videos. 
# Cross-validation showed performance variance across videos (F1 range: 
# 0.750-1.000), indicating that certain lighting conditions, 
# activity levels, or bee behaviors may challenge the classifier. Additional 
# annotation from diverse recording conditions would improve generalization."

# ✓ Results saved to /Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/ml_classifier_cv_results.csv
# (venv) edwardamoah@Edwards-MacBook-Pro-2 BeeMonitor_eai6 % 

In [25]:
"""Proper cross-validation for ML Event Classifier.

Leave-One-Video-Out cross-validation for HONEST performance metrics.
Uses current EventProcessor API (ML-first, no trajectory filtering).
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from beemonitor.core.config import Config
from beemonitor.detection.nest_detector import NestDetector
from ultralytics import YOLO
import os
import logging
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def extract_trajectory_features(centroids, action, nest_bbox):
    """Extract 20 features from trajectory for ML classification.
    
    Mirrors EventProcessor._extract_trajectory_features()
    """
    nest_x = (nest_bbox[0] + nest_bbox[2]) / 2
    nest_y = (nest_bbox[1] + nest_bbox[3]) / 2
    
    # Trajectory shape features
    trajectory_length = len(centroids)
    
    path_length = 0.0
    for i in range(len(centroids) - 1):
        dx = centroids[i+1][0] - centroids[i][0]
        dy = centroids[i+1][1] - centroids[i][1]
        path_length += np.sqrt(dx**2 + dy**2)
    
    displacement = np.sqrt(
        (centroids[-1][0] - centroids[0][0])**2 +
        (centroids[-1][1] - centroids[0][1])**2
    )
    
    tortuosity = path_length / displacement if displacement > 0 else 0
    
    # Speed profile
    speeds = []
    for i in range(len(centroids) - 1):
        dx = centroids[i+1][0] - centroids[i][0]
        dy = centroids[i+1][1] - centroids[i][1]
        speeds.append(np.sqrt(dx**2 + dy**2))
    
    avg_speed = np.mean(speeds) if speeds else 0
    max_speed = np.max(speeds) if speeds else 0
    speed_std = np.std(speeds) if speeds else 0
    speed_cv = speed_std / avg_speed if avg_speed > 0 else 0
    
    third = len(speeds) // 3 if len(speeds) >= 3 else 1
    start_speed = np.mean(speeds[:third]) if speeds else 0
    middle_speed = np.mean(speeds[third:2*third]) if len(speeds) >= 3 else avg_speed
    end_speed = np.mean(speeds[-third:]) if speeds else 0
    decel_ratio = end_speed / start_speed if start_speed > 0 else 1.0
    
    # Nest proximity
    start_to_nest = np.sqrt((centroids[0][0] - nest_x)**2 + (centroids[0][1] - nest_y)**2)
    end_to_nest = np.sqrt((centroids[-1][0] - nest_x)**2 + (centroids[-1][1] - nest_y)**2)
    approach_ratio = end_to_nest / start_to_nest if start_to_nest > 0 else 1.0
    
    # Position variance
    x_var = np.var([c[0] for c in centroids])
    y_var = np.var([c[1] for c in centroids])
    
    # Direction
    vertical_movement = centroids[-1][1] - centroids[0][1]
    horizontal_movement = abs(centroids[-1][0] - centroids[0][0])
    
    # Event type
    is_entry = 1 if action == 'Entry' else 0
    
    return {
        'trajectory_length': trajectory_length,
        'path_length': path_length,
        'displacement': displacement,
        'tortuosity': tortuosity,
        'avg_speed': avg_speed,
        'max_speed': max_speed,
        'speed_std': speed_std,
        'speed_cv': speed_cv,
        'start_speed': start_speed,
        'middle_speed': middle_speed,
        'end_speed': end_speed,
        'decel_ratio': decel_ratio,
        'start_to_nest': start_to_nest,
        'end_to_nest': end_to_nest,
        'approach_ratio': approach_ratio,
        'x_var': x_var,
        'y_var': y_var,
        'vertical_movement': vertical_movement,
        'horizontal_movement': horizontal_movement,
        'is_entry': is_entry
    }


def is_inside_bbox(point, bbox, padding=40):
    """Check if point is inside bbox with padding (matches EventProcessor)."""
    x, y = point
    x1, y1, x2, y2 = bbox
    return (x1 - padding <= x <= x2 + padding and 
            y1 - padding <= y <= y2 + padding)


def detect_events_from_tracking(tracking_df, nests):
    """Detect events from tracking data using lenient settings.
    
    Mirrors EventProcessor._detect_all_events():
    - window_size=1 (only need 1 frame inside nest)
    - padding=40 (large detection area)
    """
    events = []
    hole_bboxes = nests['nests']
    padding = 40
    
    for track_id in tracking_df['track_id'].unique():
        track_data = tracking_df[tracking_df['track_id'] == track_id].sort_values('frame')
        
        if len(track_data) < 2:
            continue
        
        centroids = []
        for _, row in track_data.iterrows():
            cx = (row['x1'] + row['x2']) / 2
            cy = (row['y1'] + row['y2']) / 2
            centroids.append((cx, cy))
        
        frame_numbers = track_data['frame'].tolist()
        
        # Check EXIT (start in nest)
        start_pos = centroids[0]
        for nest_id, bbox in hole_bboxes.items():
            if is_inside_bbox(start_pos, bbox, padding):
                events.append({
                    'action': 'Exit',
                    'nest': nest_id,
                    'frame_number': frame_numbers[0],
                    'track_id': track_id,
                    'centroids': centroids,
                    'nest_bbox': bbox
                })
                break
        
        # Check ENTRY (end in nest)
        end_pos = centroids[-1]
        for nest_id, bbox in hole_bboxes.items():
            if is_inside_bbox(end_pos, bbox, padding):
                events.append({
                    'action': 'Entry',
                    'nest': nest_id,
                    'frame_number': frame_numbers[-1],
                    'track_id': track_id,
                    'centroids': centroids,
                    'nest_bbox': bbox
                })
                break
    
    return events


def main():
    print("="*70)
    print("LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION")
    print("ML Event Classifier Evaluation")
    print("="*70)
    
    # Setup
    config = Config.default()
    nest_model = YOLO(config.models.nest_detection)
    detector = NestDetector(nest_model, config)
    
    input_data = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data"
    output_folder = "/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/CVPR_Evaluation_Video_Data_output"
    manual_csv = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/data/Manual_Foraging_Events_Observation.csv'
    
    files = os.listdir(input_data)
    files = [os.path.join(input_data, file) for file in files if 'mp4' in file]
    
    manual_df = pd.read_csv(manual_csv)
    manual_df = manual_df[['video', 'action', 'nest', 'timestamp']].dropna()
    
    def parse_manual_time(video, time_str):
        date_part = video.split('_')[1]
        return datetime.strptime(f"{date_part} {time_str}", "%Y-%m-%d %H:%M:%S")
    
    manual_df['dt'] = manual_df.apply(lambda x: parse_manual_time(x['video'], x['timestamp']), axis=1)
    
    tracking_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) 
                      if f.endswith('_tracking_results.csv')]
    tracking_data = {}
    for file in tracking_files:
        video_name = file.replace('_tracking_results.csv', '').split("/")[-1]
        tracking_data[video_name] = pd.read_csv(file)
    
    print(f"\nFound {len(tracking_data)} tracking files")
    
    # Extract features
    print("\n" + "="*70)
    print("EXTRACTING FEATURES FROM ALL EVENTS")
    print("="*70)
    
    all_features = []
    all_metadata = []  # (video_name, action, nest_id, frame)
    
    for video_name, tracking_df in tracking_data.items():
        print(f"\nProcessing {video_name}...")
        
        video_file = [f for f in files if video_name in f]
        if len(video_file) == 0:
            print(f"  Skipping - video file not found")
            continue
        
        video_file = video_file[0]
        
        try:
            nests = detector.get_nests_and_hotel_detections(video_file)
        except Exception as e:
            print(f"  Skipping - nest detection failed: {e}")
            continue
        
        # Detect events using lenient settings (matches EventProcessor)
        events = detect_events_from_tracking(tracking_df, nests)
        print(f"  Detected {len(events)} candidate events")
        
        for event in events:
            features = extract_trajectory_features(
                event['centroids'],
                event['action'],
                event['nest_bbox']
            )
            all_features.append(features)
            all_metadata.append((
                video_name, 
                event['action'], 
                event['nest'], 
                event['frame_number']
            ))
    
    features_df = pd.DataFrame(all_features)
    print(f"\n✓ Extracted features for {len(features_df)} detected events")
    
    # Match to manual labels
    print("\n" + "="*70)
    print("MATCHING TO MANUAL LABELS")
    print("="*70)
    
    labels = []
    video_groups = []
    
    for i, (video_name, action, nest_id, frame_num) in enumerate(all_metadata):
        video_groups.append(video_name)
        
        parts = video_name.split('_')
        date_str = f"{parts[1]} {parts[2]}:{parts[3]}:{parts[4]}"
        video_start = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        event_time = video_start + timedelta(seconds=frame_num/30.0)
        
        matched = False
        manual_for_video = manual_df[manual_df['video'] == video_name]
        
        for _, man_event in manual_for_video.iterrows():
            man_nest = int(man_event['nest'])
            det_nest = int(nest_id)
            
            if man_event['action'] == action and man_nest == det_nest:
                time_diff = abs(event_time - man_event['dt'])
                if time_diff <= timedelta(seconds=3.0):
                    matched = True
                    break
        
        labels.append(1 if matched else 0)
    
    labels = np.array(labels)
    video_groups = np.array(video_groups)
    
    print(f"\nLabeled {len(labels)} events:")
    print(f"  Real events (TP candidates): {sum(labels == 1)}")
    print(f"  Noise (FP candidates): {sum(labels == 0)}")
    
    if sum(labels == 1) < 10:
        print("\nERROR: Not enough positive examples for cross-validation!")
        return
    
    # LOVO Cross-Validation
    print("\n" + "="*70)
    print("LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION")
    print("="*70)
    
    unique_videos = sorted(set(video_groups))
    cv_results = []
    
    all_y_true = []
    all_y_pred = []
    
    for held_out_video in unique_videos:
        print(f"\nFold: Testing on {held_out_video}")
        
        train_mask = video_groups != held_out_video
        test_mask = video_groups == held_out_video
        
        X_train = features_df[train_mask].values
        y_train = labels[train_mask]
        X_test = features_df[test_mask].values
        y_test = labels[test_mask]
        
        if len(X_test) == 0:
            print(f"  No events in this video, skipping")
            continue
        
        if sum(y_train == 1) < 2 or sum(y_train == 0) < 2:
            print(f"  Not enough training examples, skipping")
            continue
        
        print(f"  Train: {len(X_train)} ({sum(y_train==1)} real, {sum(y_train==0)} noise)")
        print(f"  Test:  {len(X_test)} ({sum(y_test==1)} real, {sum(y_test==0)} noise)")
        
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            class_weight='balanced'
        )
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Collect for aggregate metrics
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)
        
        # Per-fold metrics
        if len(np.unique(y_test)) > 1:
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
        else:
            precision = recall = f1 = float('nan')
        
        cv_results.append({
            'video': held_out_video,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'n_test': len(X_test),
            'n_real': sum(y_test == 1),
            'tp': sum((y_pred == 1) & (y_test == 1)),
            'fp': sum((y_pred == 1) & (y_test == 0)),
            'fn': sum((y_pred == 0) & (y_test == 1)),
            'tn': sum((y_pred == 0) & (y_test == 0))
        })
        
        print(f"  P: {precision:.3f}, R: {recall:.3f}, F1: {f1:.3f}")
    
    # Summary
    print("\n" + "="*70)
    print("CROSS-VALIDATION SUMMARY")
    print("="*70)
    
    # Aggregate metrics (pooled predictions)
    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)
    
    pooled_precision = precision_score(all_y_true, all_y_pred, zero_division=0)
    pooled_recall = recall_score(all_y_true, all_y_pred, zero_division=0)
    pooled_f1 = f1_score(all_y_true, all_y_pred, zero_division=0)
    
    print(f"\nPOOLED METRICS (aggregated predictions):")
    print(f"  Precision: {pooled_precision:.3f}")
    print(f"  Recall:    {pooled_recall:.3f}")
    print(f"  F1 Score:  {pooled_f1:.3f}")
    
    # Mean per-fold metrics
    valid_results = [r for r in cv_results if not np.isnan(r['f1'])]
    
    avg_precision = np.mean([r['precision'] for r in valid_results])
    avg_recall = np.mean([r['recall'] for r in valid_results])
    avg_f1 = np.mean([r['f1'] for r in valid_results])
    
    std_precision = np.std([r['precision'] for r in valid_results])
    std_recall = np.std([r['recall'] for r in valid_results])
    std_f1 = np.std([r['f1'] for r in valid_results])
    
    print(f"\nMEAN PER-FOLD METRICS (± std):")
    print(f"  Precision: {avg_precision:.3f} ± {std_precision:.3f}")
    print(f"  Recall:    {avg_recall:.3f} ± {std_recall:.3f}")
    print(f"  F1 Score:  {avg_f1:.3f} ± {std_f1:.3f}")
    
    # Confusion matrix totals
    total_tp = sum(r['tp'] for r in cv_results)
    total_fp = sum(r['fp'] for r in cv_results)
    total_fn = sum(r['fn'] for r in cv_results)
    total_tn = sum(r['tn'] for r in cv_results)
    
    print(f"\nAGGREGATE CONFUSION MATRIX:")
    print(f"  TP: {total_tp}  FP: {total_fp}")
    print(f"  FN: {total_fn}  TN: {total_tn}")
    
    print("\nPER-VIDEO BREAKDOWN:")
    print(f"{'Video':<40} {'P':<8} {'R':<8} {'F1':<8} {'TP':<5} {'FP':<5} {'FN'}")
    print("-" * 80)
    for r in cv_results:
        p = f"{r['precision']:.3f}" if not np.isnan(r['precision']) else "N/A"
        rec = f"{r['recall']:.3f}" if not np.isnan(r['recall']) else "N/A"
        f1 = f"{r['f1']:.3f}" if not np.isnan(r['f1']) else "N/A"
        print(f"{r['video']:<40} {p:<8} {rec:<8} {f1:<8} {r['tp']:<5} {r['fp']:<5} {r['fn']}")
    
    # F1 range for paper
    valid_f1s = [r['f1'] for r in valid_results if not np.isnan(r['f1'])]
    min_f1 = min(valid_f1s) if valid_f1s else 0
    max_f1 = max(valid_f1s) if valid_f1s else 0
    
    print("\n" + "="*70)
    print("FOR YOUR RESEARCH PAPER")
    print("="*70)
    print(f"""
METHODOLOGY TEXT (Section 2.6):
"To validate the ML classifier's generalization across different recording 
conditions, we performed leave-one-video-out cross-validation. For each fold, 
the classifier was trained on trajectory features from {len(valid_results)-1} videos and 
tested on the held-out video, repeated for all {len(valid_results)} videos."

RESULTS TEXT (New Section 3.5 - ML Classifier Validation):
"The Random Forest event classifier achieved {pooled_precision:.1%} precision and 
{pooled_recall:.1%} recall (F1 = {pooled_f1:.3f}) using leave-one-video-out 
cross-validation. Mean per-fold performance was {avg_f1:.3f} ± {std_f1:.3f} F1, 
with individual video F1 scores ranging from {min_f1:.3f} to {max_f1:.3f}."

LIMITATION TEXT (Section 4.1):
"The ML event classifier requires training data from annotated videos. 
Cross-validation showed performance variance across videos (F1 range: 
{min_f1:.3f}-{max_f1:.3f}), indicating that certain lighting conditions, 
activity levels, or bee behaviors may challenge the classifier. Additional 
annotation from diverse recording conditions would improve generalization."
""")
    
    # Save results
    results_df = pd.DataFrame(cv_results)
    output_path = '/Users/edwardamoah/Documents/GitHub/BeeMonitor_eai6/output/ml_classifier_cv_results.csv'
    results_df.to_csv(output_path, index=False)
    print(f"✓ Results saved to {output_path}")


if __name__ == "__main__":
    main()

LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION
ML Event Classifier Evaluation


INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
INFO:beemonitor.detection.nest_detector:Detection attempt 1/10



Found 11 tracking files

EXTRACTING FEATURES FROM ALL EVENTS

Processing mendels_2024-05-23_18_20_01...


INFO:beemonitor.detection.nest_detector:Processing nest detections...
INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_18_20_01.png
INFO:beemonitor.detection.nest_detector:All quality checks passed
INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_18_20_01.png
INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
INFO:beemonitor.detection.nest_detector:Detection attempt 1/10


  Detected 123 candidate events

Processing mendels_2024-05-08_15_30_00...


...

  Detected 90 candidate events

Processing mendels_2024-05-23_12_00_00...


INFO:beemonitor.detection.nest_detector:Processing nest detections...
INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_00_00.png
INFO:beemonitor.detection.nest_detector:All quality checks passed
INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-05-23_12_00_00.png
INFO:beemonitor.detection.nest_detector:Starting nest detection pipeline (max attempts: 10)
INFO:beemonitor.detection.nest_detector:Detection attempt 1/10


  Detected 39 candidate events

Processing mendels_2024-04-30_09_40_01...


INFO:beemonitor.detection.nest_detector:Processing nest detections...
INFO:beemonitor.detection.nest_detector:Processed 60 nests in hotel ROI
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_40_01.png
INFO:beemonitor.detection.nest_detector:All quality checks passed
INFO:beemonitor.detection.nest_detector:Successful nest detection on attempt 1
INFO:beemonitor.detection.nest_detector:Saved nest visualization to output/nests_visualization_mendels_2024-04-30_09_40_01.png


  Detected 23 candidate events

✓ Extracted features for 610 detected events

MATCHING TO MANUAL LABELS

Labeled 610 events:
  Real events (TP candidates): 277
  Noise (FP candidates): 333

LEAVE-ONE-VIDEO-OUT CROSS-VALIDATION

Fold: Testing on mendels_2024-04-30_09_00_00
  Train: 566 (268 real, 298 noise)
  Test:  44 (9 real, 35 noise)
  P: 0.900, R: 1.000, F1: 0.947

Fold: Testing on mendels_2024-04-30_09_10_01
  Train: 566 (263 real, 303 noise)
  Test:  44 (14 real, 30 noise)
  P: 0.929, R: 0.929, F1: 0.929

Fold: Testing on mendels_2024-04-30_09_20_00
  Train: 586 (269 real, 317 noise)
  Test:  24 (8 real, 16 noise)
  P: 0.889, R: 1.000, F1: 0.941

Fold: Testing on mendels_2024-04-30_09_30_00
  Train: 566 (260 real, 306 noise)
  Test:  44 (17 real, 27 noise)
  P: 1.000, R: 0.941, F1: 0.970

Fold: Testing on mendels_2024-04-30_09_40_01
  Train: 587 (273 real, 314 noise)
  Test:  23 (4 real, 19 noise)
  P: 0.800, R: 1.000, F1: 0.889

Fold: Testing on mendels_2024-05-08_15_00_00
  Tra